In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
# Always show all columns when inspecting
pd.set_option("display.max_columns", None)

ds = load_dataset("criteo/criteo-uplift")
print(ds)

c:\Users\User\Documents\Projects\Incrementality-Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure'],
        num_rows: 13979592
    })
})


In [3]:
df = ds["train"].to_pandas()
display(df.head())

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [4]:
df = df.reset_index(drop=True)
df["user_id"] = np.arange(1,len(df)+1,dtype="int64")

cols = ["user_id"] + [col for col in df.columns if col != "user_id"]
df = df[cols]

display(df.head())

,user_id,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,1,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,2,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,3,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,4,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,5,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [5]:

print("\nShape:", df.shape)
print("\nData Types:\n", df.dtypes)


Shape: (13979592, 17)

Data Types:
 user_id         int64
f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object


In [6]:

print("Missing Values:\n", df.isna().sum().sort_values(ascending=False).head(20))

Missing Values:
 user_id       0
f0            0
f1            0
f2            0
f3            0
f4            0
f5            0
f6            0
f7            0
f8            0
f9            0
f10           0
f11           0
treatment     0
conversion    0
visit         0
exposure      0
dtype: int64


In [7]:
# Split the DataFrame into two batches for Snowflake upload (free tier limit ~250MB)
half = len(df) // 2
df1 = df.iloc[:half]
df2 = df.iloc[half:]

# Save each part as a separate Parquet file
df1.to_parquet('../data/criteo_uplift_part1.parquet', index=False)
df2.to_parquet('../data/criteo_uplift_part2.parquet', index=False)

print(f"DataFrames saved: Part 1 ({len(df1)} rows) to ../data/criteo_uplift_part1.parquet")
print(f"Part 2 ({len(df2)} rows) to ../data/criteo_uplift_part2.parquet")

DataFrames saved: Part 1 (6989796 rows) to ../data/criteo_uplift_part1.parquet
Part 2 (6989796 rows) to ../data/criteo_uplift_part2.parquet


In [8]:
display(df1.head())

,user_id,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,1,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,2,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,3,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,4,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,5,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [9]:
display(df2.head())

,user_id,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
6989796,6989797,14.354717,10.059654,8.214383,0.294543,10.280525,4.115453,-10.275874,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
6989797,6989798,25.269683,10.059654,8.214383,4.679882,10.280525,4.115453,-7.301017,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
6989798,6989799,12.616365,10.059654,8.910424,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
6989799,6989800,24.232129,10.059654,8.214383,4.679882,10.280525,4.115453,-6.699321,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
6989800,6989801,25.984048,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
